# Week 4 Discussion | Function Tool Use
### Ben Miller


## Discussion Post Prompt
Define a tool schema for a non-trivial task (weather lookup with units and location disambiguation, a database query with filters, a code-formatting tool with language and style options, or similar). Start with a deliberately loose schema, invoke the tool via an LLM, and document what the model gets wrong. Then tighten the schema (add enums, constraints, required fields) and repeat.

In [ ]:
import os
from openai import OpenAI
from google.colab import userdata
openai_api_key = userdata.get('OPENAI_API_KEY') or os.getenv("OPENAI_API_KEY")

if openai_api_key is None:
    raise ValueError("OPENAI_API_KEY not found. Please set it in Colab secrets or as an environment variable.")

openai_client = OpenAI(api_key=openai_api_key)

I used a Code Formatting function for my example.  The loosely defined tool gives unspecific instructions, and the tightly defined tool gives specific vaules for variable casing, constant casing, and indent size.

In [ ]:
LOOSE_TOOL = {
    "type": "function",
    "function": {
        "name": "format_code",
        "description": "Format a code snippet",
        "parameters": {
            "type": "object",
            "properties": {
                "code": {"type": "string"},
                "language": {"type": "string"},
                "casing": {"type": "string"},
                "indent": {"type": "string"}
            }
        }
    }
}

TIGHT_TOOL = {
    "type": "function",
    "function": {
        "name": "format_code",
        "description": "Format a code snippet's identifier casing and indentation for a given language.",
        "parameters": {
            "type": "object",
            "properties": {
                "code": {"type": "string"},
                "language": {"type": "string", "enum": ["python", "typescript", "csharp"]},
                "variable_casing": {"type": "string", "enum": ["camelCase", "snake_case", "PascalCase", "kebab-case", "hungarian"]},
                "constant_casing": {"type": "string", "enum": ["SCREAMING_SNAKE_CASE", "PascalCase", "camelCase"]},
                "indent_size": {"type": "integer", "enum": [2, 4, 8]}
            },
            "required": ["code", "language", "variable_casing"]
        }
    }
}

In [ ]:
import json

def run_format_code_test(test_prompt):
    # Run the same prompt against both the loose and tight format_code schemas.
    # Prints the input code once, then prints only the non-code parameters for
    # each schema so the loose vs. tight comparison is easy to read.
    parsed_results = {}

    for label, tool in [("LOOSE", LOOSE_TOOL), ("TIGHT", TIGHT_TOOL)]:
        response = openai_client.chat.completions.create(
            model="gpt-5.1",
            messages=[{"role": "user", "content": test_prompt}],
            tools=[tool],
            tool_choice={"type": "function", "function": {"name": "format_code"}}
        )
        args_json = response.choices[0].message.tool_calls[0].function.arguments
        parsed_results[label] = json.loads(args_json)

    # The code argument should be identical across both calls since the model
    # just passes it through, so only print it once as shared context.
    input_code = parsed_results["LOOSE"]["code"]
    print(f"Input code:\n{input_code}\n")

    for label, args in parsed_results.items():
        args_without_code = {key: value for key, value in args.items() if key != "code"}
        print(f"{label} schema parameters:\n{args_without_code}\n")

In [ ]:

PROMPT_UNAMBIGUOUS = """Convert this code to snake_case with 4-space indentation:

def CalculateTotalPrice(itemPrice, Tax_Rate):
    TotalPrice = itemPrice + (itemPrice * Tax_Rate)
    return TotalPrice
"""

PROMPT_NO_CASING_SPECIFIED = """Clean up this code:

def get_user_data(user_id, ApiKey):
    UserName = fetch_name(user_id)
    MAX_retries = 3
    return UserName
"""

PROMPT_MIXED_CASING = """Format this using camelCase for variables but keep constants in ALL CAPS:

def InitializeApp(config_options, DebugMode):
    max_connections = config_options.limit
    default_timeout = 3000
    return max_connections
"""

print("=== Unambiguous Prompt ===")
run_format_code_test(PROMPT_UNAMBIGUOUS)
print("=== No Casing Specified ===")
run_format_code_test(PROMPT_NO_CASING_SPECIFIED)
print("=== Mixed Casing ===")
run_format_code_test(PROMPT_MIXED_CASING)

=== Unambiguous Prompt ===
Input code:
def CalculateTotalPrice(itemPrice, Tax_Rate):
    TotalPrice = itemPrice + (itemPrice * Tax_Rate)
    return TotalPrice

LOOSE schema parameters:
{'language': 'python', 'casing': 'snake_case', 'indent': '    '}

TIGHT schema parameters:
{'language': 'python', 'variable_casing': 'snake_case', 'indent_size': 4}

=== No Casing Specified ===
Input code:
def get_user_data(user_id, ApiKey):
    UserName = fetch_name(user_id)
    MAX_retries = 3
    return UserName


LOOSE schema parameters:
{'language': 'python', 'casing': 'snake_case', 'indent': '4_spaces'}

TIGHT schema parameters:
{'language': 'python', 'variable_casing': 'snake_case', 'constant_casing': 'SCREAMING_SNAKE_CASE', 'indent_size': 4}

=== Mixed Casing ===
Input code:
def InitializeApp(config_options, DebugMode):
    max_connections = config_options.limit
    default_timeout = 3000
    return max_connections


LOOSE schema parameters:
{'language': 'python', 'casing': 'camelCase', 'indent':